# Clase 15: laboratorio MLOps para recomendación de cultivos

**Objetivo:** desplegar y medir el Random Forest existente con ONNX, FastAPI, Pydantic y Docker; cerrar con una estimación didáctica de Green AI.

**Proyecto existente:** Crop Recommendation, siete variables de entrada y artefactos ya guardados. No entrenaremos otro modelo ni modificaremos el dataset o los modelos originales.

**Entorno desde la raíz del repositorio (terminal VS Code / PowerShell):**

```powershell
python -m pip install -r requirements_mlops.txt onnx skl2onnx httpx
```

El archivo `requirements_mlops.txt` cubre el servicio; `onnx` y `skl2onnx` se añaden para validar y exportar desde el notebook. Selecciona en VS Code el mismo intérprete donde instalaste estos paquetes.

## 1. ¿Qué problema estamos resolviendo?

Crop Recommendation estima un cultivo adecuado a partir de siete mediciones del suelo y el ambiente: N, P, K, temperatura, humedad, pH y lluvia. El dataset procesado conserva las 22 clases de cultivo. En este repositorio hay dos Random Forest ya existentes: el modelo B2 seleccionado por `config.json` es **binario** (rice / no rice); `models/random_forest.pkl` es un clasificador histórico de **22 clases** y `models/label_encoder.pkl` convierte sus códigos a nombres. Esta práctica usa por defecto el pipeline B2 configurado; no confundiremos su salida binaria con una recomendación multiclase.

**Qué observar:** siete features, 2.200 filas del dataset y artefactos entrenados cargables. Los ejemplos de salida se obtendrán ejecutando esos modelos, no se escribirán como predicciones esperadas inventadas.

**Actividad:** ¿por qué “rice/no rice” no responde la misma pregunta que elegir entre 22 cultivos? Identifica en `config.json` cuál de los modelos está seleccionado.

## 2. ¿Qué es MLOps?

MLOps aplica prácticas de ingeniería al ciclo de vida del aprendizaje automático: versiona datos y modelos, valida entradas, empaqueta preprocesamiento y estimador, publica el servicio y monitorea su comportamiento. Aquí recorremos el flujo real del repositorio, sin volver a entrenar.

```text
datos y modelo existentes → validación → empaquetado ONNX → API → contenedor
                                               ↑                 ↓
                                   medición y monitoreo ← solicitudes
```

| Etapa | En esta práctica |
|---|---|
| Reproducibilidad | Rutas y configuración del repositorio actual |
| Empaquetado | Pipeline completo y exportación ONNX separada |
| Serving | FastAPI con validación Pydantic |
| Contenerización | Imagen Docker construida desde el Dockerfile actual |
| Observabilidad | Salud, latencia, memoria y tamaños |

**Qué observar:** despliegue no es solo crear una URL; el preprocesamiento y las dependencias deben viajar junto al modelo.

**Actividad:** señala una etapa que tendría que repetirse si cambia el modelo guardado.

## 3. Exportación a ONNX

ONNX es un formato portable de inferencia. Convertiremos el **pipeline cargado**, no solo el estimador: así el `RobustScaler` y el Random Forest quedan en el mismo grafo y la API recibe las siete variables originales. Guardaremos el resultado como `models/rf_final_b2_pipeline.onnx`; no sobrescribiremos el archivo preexistente `models/modelo.onnx`, cuya correspondencia con el bundle actual no se ha verificado.

**Qué observar:** el ONNX debe aceptar una matriz `(lote, 7)` de `float32`. El conversor y el runtime son dependencias distintas; se comprueba el grafo y se ejecuta antes de usarlo.

**Actividad:** explica por qué exportar solo `classifier` después de que el modelo espera datos escalados produciría entradas incompatibles.

In [ ]:
# Preparación del entorno y las rutas
import importlib.util
import json
import os
import socket
import shutil
import subprocess
import sys
import tempfile
import time
import uuid
from pathlib import Path

required_packages = ["onnx", "onnxruntime", "skl2onnx", "fastapi", "pydantic", "httpx"]
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]
if missing_packages:
    raise RuntimeError(
        f"Faltan dependencias: {missing_packages}. Instálalas desde la raíz con "
        "python -m pip install -r requirements_mlops.txt onnx skl2onnx httpx"
    )

import joblib
import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
import psutil
import requests
import seaborn as sns
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "processed" / "crop.parquet").exists()
     and (parent / "config.json").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró el repositorio con data/processed/crop.parquet y config.json. "
        "Abre el notebook desde el proyecto actual."
    )
os.chdir(PROJECT_ROOT)
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print("Python:", sys.version.split()[0])
for package, module_name in [
    ("scikit-learn", "sklearn"), ("onnx", "onnx"),
    ("onnxruntime", "onnxruntime"), ("skl2onnx", "skl2onnx"),
    ("fastapi", "fastapi"), ("pydantic", "pydantic"),
]:
    module = __import__(module_name)
    print(f"{package}: {getattr(module, '__version__', 'disponible')}")

### Cargar datos y modelos sin volver a entrenar

Leemos el Parquet procesado que ya existe en `data/processed/crop.parquet` y cargamos el bundle indicado por `config.json`. El bundle actual contiene el `RobustScaler` y el Random Forest binario. Esta celda no vuelve a entrenar, reescala, guardar ni reemplazar ningún modelo, configuración o dato.

In [ ]:
# Cargamos exactamente el bundle de inferencia seleccionado por config.json.
CONFIG_PATH = PROJECT_ROOT / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
FEATURES = CONFIG["feature_columns"]
MODE = str(CONFIG.get("mode", "binary")).lower()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "crop.parquet"
MODEL_BUNDLE_PATH = PROJECT_ROOT / CONFIG.get(
    "model_bundle_path", "models/rf_final_b2_pipeline.pkl"
)
MULTICLASS_MODEL_PATH = PROJECT_ROOT / "models" / "random_forest.pkl"
LABEL_ENCODER_PATH = PROJECT_ROOT / "models" / "label_encoder.pkl"

if len(FEATURES) != 7:
    raise ValueError(f"Se esperaban siete variables; config contiene {len(FEATURES)}.")
if not DATA_PATH.exists() or not MODEL_BUNDLE_PATH.exists():
    raise FileNotFoundError(f"No se encontró el Parquet o bundle: {DATA_PATH}, {MODEL_BUNDLE_PATH}")

df = pd.read_parquet(DATA_PATH)
if not set(FEATURES + ["label"]).issubset(df.columns):
    raise ValueError("El Parquet no contiene las siete variables y la columna label.")
X_raw = df[FEATURES].copy()
y = df["label"].astype(str)
serving_model = joblib.load(MODEL_BUNDLE_PATH)
if not hasattr(serving_model, "predict_proba") or not hasattr(serving_model, "classes_"):
    raise TypeError("El bundle existente debe exponer predict_proba y classes_.")
if list(getattr(serving_model, "feature_names_in_", FEATURES)) != FEATURES:
    raise ValueError("El orden de features del bundle no coincide con config.json.")
if MODE == "binary" and set(serving_model.classes_) != {0, 1}:
    raise ValueError("config.json declara modo binario, pero el bundle no tiene clases 0 y 1.")

# También comprobamos, sin cambiarlo, el RF multiclase histórico y su decodificador.
multiclass_model = joblib.load(MULTICLASS_MODEL_PATH)
label_encoder = joblib.load(LABEL_ENCODER_PATH)
if len(multiclass_model.classes_) != len(label_encoder.classes_):
    raise ValueError("El RF multiclase y su LabelEncoder no tienen la misma cantidad de clases.")
first_row = X_raw.iloc[[0]]
multiclass_id = int(multiclass_model.predict(first_row)[0])
multiclass_name = str(label_encoder.inverse_transform([multiclass_id])[0])
multiclass_confidence = float(np.max(multiclass_model.predict_proba(first_row)[0]))

print(f"Datos: {df.shape}; features: {FEATURES}")
print(f"Bundle activo: {MODEL_BUNDLE_PATH.name}; modo: {MODE}; clases: {list(serving_model.classes_)}")
print(f"Predicción real del RF multiclase para la primera fila: {multiclass_name} ({multiclass_confidence:.3f})")
print("Los modelos, el encoder y los datos solo se leyeron; no se reescribieron.")

### Convertir el pipeline completo

La conversión usa las siete columnas originales como entrada. ONNX contiene tanto el escalador como el clasificador, y el artefacto se guarda en una ruta nueva para preservar `models/modelo.onnx` y todos los modelos existentes.

**Qué observar:** el modelo generado es derivado del pipeline ya cargado; no se ajustan parámetros ni se entrena un estimador nuevo.

In [ ]:
# Convertimos la tubería completa, incluido el RobustScaler.
ONNX_PATH = PROJECT_ROOT / "models" / "rf_final_b2_pipeline.onnx"
classifier = (
    serving_model.named_steps["classifier"]
    if hasattr(serving_model, "named_steps")
    else serving_model
)
onnx_model = convert_sklearn(
    serving_model,
    initial_types=[("input", FloatTensorType([None, len(FEATURES)]))],
    options={id(classifier): {"zipmap": False}},
    target_opset=17,
)
onnx.checker.check_model(onnx_model)
ONNX_PATH.write_bytes(onnx_model.SerializeToString())
onnx_session = ort.InferenceSession(
    str(ONNX_PATH), providers=["CPUExecutionProvider"]
)
onnx_input = onnx_session.get_inputs()[0]
if onnx_input.shape[1] != len(FEATURES):
    raise AssertionError(f"ONNX tiene forma de entrada inesperada: {onnx_input.shape}")
onnx_input_name = onnx_input.name
print(f"ONNX validado y guardado en un archivo nuevo: {ONNX_PATH}")
print("Entrada ONNX:", onnx_input.name, onnx_input.shape, onnx_input.type)
print("Salidas ONNX:", [(item.name, item.shape, item.type) for item in onnx_session.get_outputs()])

## 4. Comparar predicciones: scikit-learn vs ONNX

Comparamos todas las filas del dataset usando la misma entrada `float32`, que es el tipo que recibe ONNX. Comprobamos etiquetas y decisiones con el umbral configurado. Las probabilidades pueden diferir de forma medible en algunas filas por el redondeo del escalado y los umbrales de los ?rboles; mostramos la diferencia m?xima en vez de exigir igualdad num?rica falsa. Esta comprobaci?n de decisiones solo cubre el umbral configurado.

In [ ]:
# Comparamos todas las filas con el mismo tipo y orden de entrada.
X_compare = X_raw.astype(np.float32)
X_compare_array = X_compare.to_numpy(dtype=np.float32)
sklearn_labels = np.asarray(serving_model.predict(X_compare)).reshape(-1)
sklearn_probabilities = np.asarray(serving_model.predict_proba(X_compare), dtype=float)
onnx_outputs = onnx_session.run(None, {onnx_input_name: X_compare_array})
onnx_labels = np.asarray(onnx_outputs[0]).reshape(-1)
onnx_probability_output = onnx_outputs[1]

# ZipMap produce diccionarios; zipmap=False normalmente produce un tensor.
if isinstance(onnx_probability_output, list) and onnx_probability_output and isinstance(onnx_probability_output[0], dict):
    onnx_probabilities = np.asarray(
        [[row.get(cls, row.get(str(cls))) for cls in serving_model.classes_]
         for row in onnx_probability_output], dtype=float
    )
else:
    onnx_probabilities = np.asarray(onnx_probability_output, dtype=float)

labels_match = np.array_equal(sklearn_labels.astype(str), onnx_labels.astype(str))
positive_index = list(serving_model.classes_).index(1)
comparison_threshold = float(CONFIG.get("threshold", 0.5))
sklearn_decisions = sklearn_probabilities[:, positive_index] >= comparison_threshold
onnx_decisions = onnx_probabilities[:, positive_index] >= comparison_threshold
decisions_match = np.array_equal(sklearn_decisions, onnx_decisions)
probabilities_valid = (
    np.isfinite(onnx_probabilities).all()
    and np.all((onnx_probabilities >= 0) & (onnx_probabilities <= 1))
    and np.allclose(onnx_probabilities.sum(axis=1), 1.0, atol=1e-6)
)
max_probability_error = float(np.max(np.abs(sklearn_probabilities - onnx_probabilities)))
print("Filas comparadas:", len(X_compare))
print("Etiquetas coincidentes:", labels_match)
print(f"Decisiones coincidentes con umbral {comparison_threshold:.2f}:", decisions_match)
print(f"M?xima diferencia absoluta de probabilidades: {max_probability_error:.8g}")
print("Probabilidades ONNX finitas, en [0, 1] y normalizadas:", probabilities_valid)
if not labels_match:
    raise AssertionError("ONNX no conserv? las etiquetas de scikit-learn.")
if not decisions_match:
    raise AssertionError("ONNX cambi? decisiones al umbral configurado.")
if not probabilities_valid:
    raise AssertionError("ONNX produjo probabilidades inv?lidas o no normalizadas.")

### Latencia de inferencia comparable

Medimos solicitudes de una fila con idénticas siete entradas, 20 llamadas de calentamiento y cinco rondas de 100 repeticiones por motor. Guardamos el tiempo de cada solicitud y reportamos media, mediana, p95 y p99. La comparación es local y no es una promesa de rendimiento de producción.

**Qué observar:** la media es sensible a valores extremos; p95/p99 describen la cola lenta. El tiempo cambia con CPU, carga, versiones y tamaño del lote.

**Actividad:** ¿qué sesgo aparecería si se midiera ONNX con lotes y scikit-learn con una fila por llamada?

In [ ]:
# Calentamos ambos motores y medimos de forma alternada, una fila por llamada.
latency_rows = X_raw.iloc[:100]
latency_arrays = latency_rows.to_numpy(dtype=np.float32)
latency_frames = [latency_rows.iloc[[index]] for index in range(len(latency_rows))]

def sklearn_inference(one_row):
    return serving_model.predict(one_row), serving_model.predict_proba(one_row)

def onnx_inference(one_row):
    return onnx_session.run(None, {onnx_input_name: one_row})

for index in range(20):
    sklearn_inference(latency_frames[index % len(latency_frames)])
    row_index = index % len(latency_arrays)
    onnx_inference(latency_arrays[row_index:row_index + 1])

latency_measurements_ms = {"Scikit-learn": [], "ONNX Runtime": []}
for repetition in range(5):
    engines = ["Scikit-learn", "ONNX Runtime"]
    if repetition % 2:
        engines.reverse()
    for engine in engines:
        for index in range(100):
            started = time.perf_counter_ns()
            if engine == "Scikit-learn":
                sklearn_inference(latency_frames[index])
            else:
                onnx_inference(latency_arrays[index:index + 1])
            latency_measurements_ms[engine].append((time.perf_counter_ns() - started) / 1e6)

latency_statistics = {}
for engine, values in latency_measurements_ms.items():
    values = np.asarray(values, dtype=float)
    latency_statistics[engine] = {
        "mean_ms": float(np.mean(values)),
        "median_ms": float(np.median(values)),
        "p95_ms": float(np.percentile(values, 95)),
        "p99_ms": float(np.percentile(values, 99)),
    }
latency_table = pd.DataFrame(latency_statistics).T
print("Repeticiones medidas por motor:", len(latency_measurements_ms["Scikit-learn"]))
display(latency_table.round(4))

## 5. FastAPI: consultar el modelo

FastAPI expone la inferencia como servicio HTTP y genera Swagger UI automáticamente. La API carga el bundle una vez al iniciar, indica si usa ONNX Runtime o scikit-learn en `/health` y devuelve predicción, clase y confianza en `/predict`.

Para probar Swagger desde una **terminal de VS Code abierta en la raíz del repositorio**:

```powershell
python -m uvicorn app.api:app --host 127.0.0.1 --port 8000 --reload
```

Abre [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs), ejecuta `GET /health` y luego `POST /predict`. Detén el servidor con `Ctrl+C`.

**Actividad:** cambia una medición válida en Swagger y observa la respuesta real; no interpretes la confianza como garantía agronómica.

In [ ]:
# Preparamos el cuerpo HTTP desde una fila real del Parquet.
sample_payload = {
    name: float(X_raw.iloc[0][name])
    for name in FEATURES
}
invalid_payload = {**sample_payload, "humidity": 1000.0}
print("Cuerpo JSON válido:")
print(json.dumps(sample_payload, indent=2))
print("Cuerpo con humedad fuera del rango permitido:")
print(json.dumps(invalid_payload, indent=2))
print("API implementada en:", PROJECT_ROOT / "app" / "api.py")
if not (PROJECT_ROOT / "app" / "api.py").exists():
    raise FileNotFoundError("No se encontró app/api.py.")

### Peticiones HTTP reales desde el notebook

La celda siguiente levanta temporalmente el mismo `app.api:app` en un puerto libre y realiza solicitudes HTTP con `requests` (no usa `TestClient`). Comprueba `/health`, predicción válida y errores 422 por valor fuera de rango, campo faltante y campo extra; al terminar, detiene el servidor temporal.

El servidor interactivo de Swagger se inicia aparte con el comando anterior y escucha en el puerto 8000.

In [ ]:
# Prueba el servidor FastAPI real mediante HTTP y lo cierra al finalizar.
with socket.socket() as port_socket:
    port_socket.bind(("127.0.0.1", 0))
    api_test_port = port_socket.getsockname()[1]
api_test_url = f"http://127.0.0.1:{api_test_port}"
api_command = [
    sys.executable, "-m", "uvicorn", "app.api:app",
    "--host", "127.0.0.1", "--port", str(api_test_port),
]
with tempfile.TemporaryFile(mode="w+b") as server_log:
    server_process = subprocess.Popen(
        api_command, cwd=PROJECT_ROOT, stdout=server_log, stderr=subprocess.STDOUT
    )
    try:
        health_response = None
        for _ in range(40):
            if server_process.poll() is not None:
                server_log.flush()
                server_log.seek(0)
                log_text = server_log.read().decode("utf-8", errors="replace")
                raise RuntimeError(f"Uvicorn terminó al iniciar:\n{log_text}")
            try:
                health_response = requests.get(f"{api_test_url}/health", timeout=1)
                break
            except (requests.ConnectionError, requests.Timeout):
                time.sleep(0.5)
        if health_response is None:
            server_log.flush()
            server_log.seek(0)
            log_text = server_log.read().decode("utf-8", errors="replace")
            raise TimeoutError(f"La API no respondió a tiempo:\n{log_text}")

        prediction_response = requests.post(
            f"{api_test_url}/predict", json=sample_payload, timeout=10
        )
        range_error = requests.post(
            f"{api_test_url}/predict", json=invalid_payload, timeout=10
        )
        missing_field_payload = dict(sample_payload)
        missing_field_payload.pop("ph")
        missing_field_error = requests.post(
            f"{api_test_url}/predict", json=missing_field_payload, timeout=10
        )
        extra_field_payload = {**sample_payload, "unexpected": 1}
        extra_field_error = requests.post(
            f"{api_test_url}/predict", json=extra_field_payload, timeout=10
        )

        print("GET /health:", health_response.status_code, health_response.json())
        print("POST /predict válido:", prediction_response.status_code, prediction_response.json())
        print("POST /predict humidity=1000:", range_error.status_code, range_error.json())
        print("POST /predict sin ph:", missing_field_error.status_code, missing_field_error.json())
        print("POST /predict con campo extra:", extra_field_error.status_code, extra_field_error.json())
        assert health_response.status_code == 200
        assert prediction_response.status_code == 200
        assert 0.0 <= prediction_response.json()["confidence"] <= 1.0
        assert range_error.status_code == 422
        assert missing_field_error.status_code == 422
        assert extra_field_error.status_code == 422
    finally:
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            server_process.kill()
            server_process.wait(timeout=10)
print("Servidor temporal detenido.")

## 6. Pydantic: entradas válidas e inválidas

`InputData` exige las siete variables, límites plausibles y rechaza campos no declarados y valores no finitos. FastAPI transforma los errores de validación en HTTP **422** con detalles del campo; la celda anterior los verificó mediante peticiones HTTP reales.

Ejemplos para Swagger:

- **Válido:** el JSON de la celda anterior, construido desde una fila real.
- **Fuera de rango:** cambiar `humidity` a `1000`.
- **Faltante:** eliminar `ph`.
- **Tipo incorrecto:** enviar `"rainfall": "lluvia"`.
- **Campo extra:** añadir `"unexpected": 1`.

**Qué observar:** el servidor no llama al modelo con solicitudes que no cumplen el esquema. La validación evita errores de formato, pero no detecta automáticamente observaciones fuera de la distribución de entrenamiento.

**Actividad:** ejecuta en `/docs` dos ejemplos inválidos y compara el cuerpo del 422. ¿Qué campo identifica cada detalle?

## 7. Docker: construir, ejecutar y comprobar

La imagen usa el Dockerfile y los artefactos existentes del repositorio; no crea un proyecto paralelo. La siguiente celda intenta la secuencia completa: comprobar cliente y daemon Docker, construir la imagen, ejecutar un contenedor, consultar `/health`, enviar una predicción real y detenerlo.

**Requisito:** Docker Desktop instalado y en ejecución, con acceso a descargar `python:3.10-slim` si la imagen base no está en caché. Si Docker no está disponible en esta máquina, la celda lo marcará como **no verificado**, sin fingir que el contenedor se probó.

**Actividad:** compara el `runtime` de `/health` en el contenedor y en la API local.

In [ ]:
# Construye y prueba el contenedor si Docker CLI y el daemon responden.
DOCKER_VALIDATION = "not_available"
docker_executable = shutil.which("docker")
if docker_executable is None:
    print("NO VERIFICADO: no se encontró Docker CLI. Instala/abre Docker Desktop y repite esta celda.")
else:
    daemon_check = subprocess.run(
        [docker_executable, "info", "--format", "{{.ServerVersion}}"],
        capture_output=True, text=True, check=False,
    )
    if daemon_check.returncode != 0:
        DOCKER_VALIDATION = "daemon_not_ready"
        print("NO VERIFICADO: Docker CLI existe, pero el daemon no responde.")
        print(daemon_check.stderr.strip())
    else:
        DOCKER_VALIDATION = "running"
        suffix = uuid.uuid4().hex[:8]
        image_tag = f"crop-api-class15:{suffix}"
        container_name = f"crop-api-class15-{suffix}"
        with socket.socket() as port_socket:
            port_socket.bind(("127.0.0.1", 0))
            docker_host_port = port_socket.getsockname()[1]

        build_result = subprocess.run(
            [docker_executable, "build", "-t", image_tag, "."],
            cwd=PROJECT_ROOT, capture_output=True, text=True, check=False,
        )
        if build_result.returncode != 0:
            DOCKER_VALIDATION = "build_failed"
            raise RuntimeError(
                "Falló docker build:\n" + "\n".join(build_result.stderr.splitlines()[-30:])
            )
        print("Imagen construida:", image_tag)

        run_result = subprocess.run(
            [docker_executable, "run", "--detach", "--name", container_name,
             "-p", f"127.0.0.1:{docker_host_port}:8000", image_tag],
            cwd=PROJECT_ROOT, capture_output=True, text=True, check=False,
        )
        if run_result.returncode != 0:
            DOCKER_VALIDATION = "container_start_failed"
            raise RuntimeError(f"Falló docker run:\n{run_result.stderr.strip()}")

        docker_url = f"http://127.0.0.1:{docker_host_port}"
        try:
            container_health = None
            for _ in range(60):
                try:
                    container_health = requests.get(f"{docker_url}/health", timeout=1)
                    break
                except (requests.ConnectionError, requests.Timeout):
                    time.sleep(1)
            if container_health is None:
                logs = subprocess.run(
                    [docker_executable, "logs", container_name],
                    capture_output=True, text=True, check=False,
                )
                raise TimeoutError(f"El contenedor no respondió:\n{logs.stdout}\n{logs.stderr}")

            container_prediction = requests.post(
                f"{docker_url}/predict", json=sample_payload, timeout=10
            )
            container_invalid = requests.post(
                f"{docker_url}/predict", json=invalid_payload, timeout=10
            )
            print("GET /health en contenedor:", container_health.status_code, container_health.json())
            print("POST /predict en contenedor:", container_prediction.status_code, container_prediction.json())
            print("POST /predict inválido en contenedor:", container_invalid.status_code)
            assert container_health.status_code == 200
            assert container_prediction.status_code == 200
            assert container_invalid.status_code == 422
        finally:
            stop_result = subprocess.run(
                [docker_executable, "stop", container_name],
                capture_output=True, text=True, check=False,
            )
            if stop_result.returncode != 0:
                raise RuntimeError(f"Falló docker stop:\n{stop_result.stderr.strip()}")
            print("Contenedor detenido:", container_name)
        DOCKER_VALIDATION = "verified"

### Secuencia manual en dos terminales de VS Code

Desde la raíz del repositorio, con Docker Desktop iniciado:

```powershell
docker version
docker build -t crop-api:class15 .
docker run --name crop-api-class15 -p 8000:8000 crop-api:class15
```

Deja esa terminal activa. En otra terminal, verifica el servicio y predice con una fila real del Parquet:

```powershell
Invoke-RestMethod http://127.0.0.1:8000/health
$body = '{"N":90,"P":42,"K":43,"temperature":20.87974371,"humidity":82.00274423,"ph":6.502985292,"rainfall":202.9355362}'
Invoke-RestMethod -Method Post -Uri http://127.0.0.1:8000/predict -ContentType 'application/json' -Body $body
```

Abre [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs) para Swagger. Cuando termines, detén el contenedor desde la segunda terminal:

```powershell
docker stop crop-api-class15
docker rm crop-api-class15
```

**Qué observar:** `/health` debe mostrar `status: ok`, el motor cargado y siete features. En `/predict`, compara clase y confianza; una entrada inválida debe responder 422.

## 8. Profiling: latencia, RAM y tamaño

Comparamos métricas obtenidas en esta ejecución:

- **Media y mediana (ms/solicitud):** tiempo típico; la mediana resiste mejor valores extremos.
- **p95 y p99:** latencia bajo la cual terminaron el 95 % y el 99 % de las llamadas medidas; útiles para observar colas lentas.
- **RAM residente (MB):** RSS observado en procesos aislados después de cargar y calentar un motor. Incluye runtime y Python, así que no es el tamaño puro del estimador.
- **Tamaño en disco (MB):** bytes del bundle scikit-learn y el ONNX; no equivale a RAM.

Las pruebas usan la misma muestra y llamadas individuales. Las cifras varían según equipo y versiones; no se rellenan con valores históricos. Los gráficos salen de mediciones locales, no de predicciones energéticas.

In [ ]:
# Medimos RAM en procesos separados y graficamos métricas observadas.
worker_code = r"""
import json, os, sys
import joblib
import numpy as np
import psutil

engine, artifact_path, row_json = sys.argv[1:]
process = psutil.Process(os.getpid())
baseline_bytes = process.memory_info().rss
row = np.asarray([json.loads(row_json)], dtype=np.float32)
if engine == "sklearn":
    model = joblib.load(artifact_path)
    model.predict(row)
    model.predict_proba(row)
else:
    import onnxruntime as ort
    session = ort.InferenceSession(artifact_path, providers=["CPUExecutionProvider"])
    session.run(None, {session.get_inputs()[0].name: row})
rss_bytes = process.memory_info().rss
print(json.dumps({
    "rss_mb": rss_bytes / (1024 ** 2),
    "incremental_mb": (rss_bytes - baseline_bytes) / (1024 ** 2),
    "baseline_mb": baseline_bytes / (1024 ** 2),
}))
"""

def measure_isolated_rss(engine, artifact_path, row):
    result = subprocess.run(
        [sys.executable, "-c", worker_code, engine, str(artifact_path), json.dumps(row)],
        cwd=PROJECT_ROOT, capture_output=True, text=True, check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"Falló la medición de RAM de {engine}:\n{result.stderr}")
    return json.loads(result.stdout.strip().splitlines()[-1])

sklearn_ram = measure_isolated_rss(
    "sklearn", MODEL_BUNDLE_PATH, X_raw.iloc[0].to_numpy(dtype=np.float32).tolist()
)
onnx_ram = measure_isolated_rss(
    "onnx", ONNX_PATH, X_raw.iloc[0].to_numpy(dtype=np.float32).tolist()
)
ram_mb = {"Scikit-learn": sklearn_ram["rss_mb"], "ONNX Runtime": onnx_ram["rss_mb"]}
size_mb = {
    "Scikit-learn": MODEL_BUNDLE_PATH.stat().st_size / (1024 ** 2),
    "ONNX Runtime": ONNX_PATH.stat().st_size / (1024 ** 2),
}
profile_table = latency_table.copy()
profile_table["rss_mb"] = pd.Series(ram_mb)
profile_table["artifact_mb"] = pd.Series(size_mb)
display(profile_table.round(4))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
latency_table[["mean_ms", "median_ms"]].plot(kind="bar", ax=axes[0, 0])
axes[0, 0].set_title("Latencia por solicitud")
axes[0, 0].set_ylabel("milisegundos")
axes[0, 0].tick_params(axis="x", rotation=0)
for engine, values in latency_measurements_ms.items():
    axes[0, 1].plot(
        [50, 95, 99], np.percentile(values, [50, 95, 99]), marker="o", label=engine
    )
axes[0, 1].set_title("Percentiles de latencia")
axes[0, 1].set_xlabel("percentil")
axes[0, 1].set_ylabel("milisegundos")
axes[0, 1].legend()
pd.Series(ram_mb).plot(kind="bar", ax=axes[1, 0], color=["#4C78A8", "#F58518"])
axes[1, 0].set_title("RSS de proceso aislado después de cargar")
axes[1, 0].set_ylabel("MB")
axes[1, 0].tick_params(axis="x", rotation=0)
pd.Series(size_mb).plot(kind="bar", ax=axes[1, 1], color=["#4C78A8", "#F58518"])
axes[1, 1].set_title("Tamaño real de los artefactos en disco")
axes[1, 1].set_ylabel("MB")
axes[1, 1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()
print("RAM RSS total (MB):", {key: round(value, 3) for key, value in ram_mb.items()})
print("Incremento aproximado sobre el proceso base (MB):", {
    "Scikit-learn": round(sklearn_ram["incremental_mb"], 3),
    "ONNX Runtime": round(onnx_ram["incremental_mb"], 3),
})

## 9. Green AI: simulación transparente

La potencia, el tiempo encendido, la cantidad de solicitudes y el factor de emisión son controles de una **simulación hipotética**. No usamos mediciones eléctricas del equipo ni convertimos la latencia en energía.

- Energía total (kWh) = potencia estimada (W) / 1.000 × horas.
- CO₂e estimado (kg) = energía total × factor de emisión (kg CO₂e/kWh).
- Energía y CO₂e por predicción = total / número de predicciones indicado.

El cálculo no atribuye automáticamente toda la energía del equipo al modelo: es una aproximación docente que depende de los supuestos introducidos.

**Qué observar:** más potencia o más horas elevan la energía total; repartirla entre más predicciones reduce la estimación por predicción, pero no cambia el consumo total asumido.

**Actividad:** ajusta el factor de emisión de tu región y compara el resultado sin presentarlo como medición real.

In [ ]:
# Controles interactivos: todos los resultados se etiquetan como estimaciones.
import ipywidgets as widgets

def estimate_green_ai(power_watts, operating_hours, predictions, emission_factor):
    if power_watts < 0 or operating_hours < 0 or predictions <= 0 or emission_factor < 0:
        raise ValueError("Potencia, horas y factor deben ser no negativos; predicciones debe ser mayor que cero.")
    energy_kwh = (power_watts / 1000) * operating_hours
    co2_kg = energy_kwh * emission_factor
    print("ESTIMACIÓN HIPOTÉTICA — no es una medición del equipo")
    print(f"Energía total: {energy_kwh:.6f} kWh")
    print(f"CO₂e total: {co2_kg:.6f} kg")
    print(f"Energía por predicción: {energy_kwh / predictions:.9f} kWh")
    print(f"CO₂e por predicción: {co2_kg / predictions:.9f} kg")

widgets.interact(
    estimate_green_ai,
    power_watts=widgets.FloatSlider(value=65, min=1, max=500, step=5, description="Potencia W"),
    operating_hours=widgets.FloatSlider(value=2, min=0.1, max=24, step=0.1, description="Horas"),
    predictions=widgets.IntSlider(value=10000, min=1, max=1000000, step=1000, description="Predicciones"),
    emission_factor=widgets.FloatSlider(value=0.4, min=0, max=1.5, step=0.01, description="kg CO₂e/kWh"),
)

## 10. Actividad final para estudiantes

En parejas o individualmente, entreguen:

1. El modo del modelo activo y la diferencia entre el RF binario B2 y el RF multiclase histórico.
2. Una tabla de tres peticiones válidas con variables distintas, predicción y confianza observadas desde `/docs`.
3. Un ejemplo inválido y la explicación del 422 de Pydantic.
4. Resultado de comparación de etiquetas/probabilidades y gráfico de latencia/RAM/tamaño.
5. Evidencia de Docker: salida de `/health`, respuesta de `/predict` y contenedor detenido. Si Docker no está instalado, anoten el requisito pendiente; no reporten esta etapa como completada.
6. Una estimación Green AI con sus cuatro supuestos claramente identificados.

**Pregunta de cierre:** ¿qué medición adicional necesitarías para convertir la simulación de Green AI en una medición de energía defendible?

## Conclusiones

- El artefacto activo y su preprocesamiento se mantienen unidos; ONNX se compara antes de servirlo.
- La API valida siete entradas y responde según el modo binario o multiclase, sin convertir códigos de clase de forma incorrecta.
- Latencia, RAM residente y tamaños son mediciones de esta ejecución; varían por entorno.
- Los números de Green AI son estimaciones explícitas basadas en supuestos del estudiante.
- Docker solo se considera verificado si se construyó la imagen y las peticiones HTTP se hicieron al contenedor.

In [ ]:
print("Laboratorio terminado.")
print("Modo del API según config.json:", MODE)
print("ONNX comparado:", ONNX_PATH.exists())
print("Verificación Docker:", DOCKER_VALIDATION)